In [1]:
from openai import OpenAI
from groq import Groq
from dotenv import load_dotenv
from pathlib import Path
from IPython.display import Markdown, display
import os 
from builders.pdf_extractor import extract_text_pdf
from rich import print as rprint
import json

In [2]:
load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

In [3]:
# models
gpt_oss = "openai/gpt-oss-120b"

In [4]:
def openai_llm_model(base_url, api_key, messages, model):
    client = OpenAI(base_url= base_url,  api_key=api_key)
    response = client.chat.completions.create(
    messages=messages,
    model=model,
    # stream= True,
    temperature=0
    )

    return response.choices[0].message.content

    

In [5]:
def groq_llm_model(messages: str, model: str, base_url: str = None, api_key : str = None):
    client = Groq(base_url= base_url,  api_key=api_key)
    response = client.chat.completions.create(
    messages=messages,
    model=model,
    # stream= True,
    temperature=0
    )

    return response.choices[0].message.content

In [6]:
resume_data = extract_text_pdf("./resume_ark/resume_original.pdf")

In [7]:
json_schema = {}
with open(r"./schema/resume_schema.json", "r") as schema:
    json_schema = json.load(schema)

In [8]:
job_description_ts = """ 
Trumid is a dynamic fintech revolutionizing the landscape of fixed income trading. With intelligent, easy-to-use, electronic solutions, we are rapidly growing and seeking exceptional talent to help redefine the boundaries of technology and finance.

Founded in 2014 by a team of fixed income market experts, Trumid has quickly become one of the top three corporate bond e-trading platforms in the U.S. Today, over 1,300 traders from an extensive and expanding client network of 890+ buy-and sell-side institutions transact on Trumid monthly.

With a rich history of innovation and a unique ability to innovate at scale, we collaborate closely with our clients, iterating quickly toward optimal solutions. With market share and client engagement at all-time highs and our pace of product development faster than ever, this is an exciting and transformative time at Trumid.

Our business model thrives on participation, and so does our company culture. We rely on every team member’s contribution to help us accomplish our goals. To succeed at Trumid, you must be curious, passionate about your craft, ambitious, collaborative, and driven.

Learn more at www.trumid.com.

The opportunity.

Are you ready to make a real impact at a fast-paced, innovative fintech company? Trumid, a rapidly growing New York city-based leader in fixed income trading technology, is looking for a driven AI Development Intern to join our team for Summer 2026. This is a paid, 8–10 week internship where you’ll work at the intersection of finance and technology, gaining hands-on experience while contributing to our growth. The ideal candidate has the following skills:

About you.

Within two years of undergraduate graduation (December 2026 – May 2028 graduation date).
AI-Native. You don’t just use AI tools; you think in them.
Intellectually Curious. You go down rabbit holes. You ask “why” until you actually understand.
Resilient & Proactive. When you hit a wall, you try to find a door before asking for help. You don’t wait to be told what to do next.
A Norm Challenger. You see a clunky process and immediately start thinking about how to make it better.
An Entrepreneur at Heart. You think about ownership, impact, and scale, not just the task in front of you.
Someone Who Knows Work Can Be Fun. You bring energy and genuine enthusiasm.
What You’ll Do.

Build real things with Claude Code. You’ll use AI-native development tools to engineer actual features and enhancements.
Come up with ideas and prototype them. Got a spark of an idea? Build a rough version and show us. We love interns who bring concepts nobody asked for but everyone immediately wants.
Learn our AI workflows — then help improve them. Get up to speed on how our team builds with AI today, then tell us what could be better. Your fresh perspective is genuinely valuable.
Be our AI radar. Read new research, follow emerging tools, and share what you find. Found something cool? Tell the team. We want your curiosity, not just your code.
Own a real problem. You’ll get pointed at an actual internal workflow and given the space to figure out how AI can make it faster or smarter. Some guidance included, but you’ll be driving.
Ideal Candidate

Able to work EST hours from New York, or remotely.
Currently pursuing a degree in Computer Science, AI/ML, Data Science, or a related STEM field.
Hands-on experience with Claude Code (Cursor, Codex, or similar).
Basic experience with Python and/or TypeScript; GitHub experience is a plus.
A side project, class project, or anything you built outside of coursework that you can talk about.
Self-sufficient enough to get unstuck on your own before asking for help.
Why Trumid?

We are operating at the edge of AI-driven engineering.
Gain experience with cutting-edge fintech technology.
Work in a collaborative, high-energy team.
Develop skills that will help you succeed in the finance and technology sectors.
Benefit from direct mentorship and guidance from experienced marketing and communications professionals.
In compliance with New York City Pay Transparency Law, the hourly pay range for this role in New York City is between $30 – $38 per hour. Several factors are considered when determining a candidate’s compensation, including academic standing, relevant experience, and qualifications. Please note that the hourly range listed for this position is based on the level of experience outlined in the job description. If a candidate’s experience differs from the requirements, the hourly rate may be adjusted accordingly.

Trumid is an equal opportunity employer.
"""

In [9]:
system_prompt = f"""
You are an elite recruitment advisor and career coach with over 20 years of experience in talent acquisition, 
resume writing, and career development across diverse industries. You specialize in tailoring resumes to specific 
job descriptions, ensuring they are ATS-optimized and positioned to stand out in a competitive candidate pool.

The user will provide you with:
1. Their current resume text.
2. A target job description they want to apply for.

Your Task:
You are NOT an advisor giving suggestions — you are actively rewriting and tailoring the resume.
Your output must be a single, valid JSON object that strictly follows the schema provided below.
Do not include any text, explanation, or commentary outside of the JSON response.

How to tailor the resume:
- Rewrite the resume content so it is fully aligned with the provided job description.
- Naturally incorporate relevant keywords from the job posting to ensure ATS compatibility.
- Maintain a professional, achievement-focused tone throughout.
- Lead with a compelling Professional Summary that directly mirrors the language and priorities of the job description.
- Prioritize and highlight the candidate's most relevant skills, experiences, and accomplishments.
- Remove or minimize details that are unrelated to the target role.
- Use strong action verbs and quantify results wherever the existing data supports it.
- Reorder sections where necessary to front-load the most relevant qualifications.
- Suggest 2-3 bullet points the candidate could add to strengthen their application.

When processing the job description, you must:
1. Extract the job title, company name, required skills, qualifications, responsibilities, and emphasized keywords.
2. Identify the company's tone and culture from the language used in the posting.
3. Use that analysis to inform every section of the rewritten resume.

Content Rules:
- You may rephrase, reframe, or combine existing details to better match the role.
- If the job description cannot be read or is unclear, return a JSON response with an appropriate error message in the relevant field.
- Only process content related to career development and job applications.

Output Rules:
- Your response must be a single valid JSON object only — no markdown, no code fences, no explanation.
- Follow the exact JSON schema provided below. Do not change or rename any keys.
- You may only change the values to reflect the tailored resume content.

JSON Schema:
{str(json_schema)}
"""

In [10]:
user_prompt = f""" 
Here is my current resume:
<resume>
{resume_data}
</resume>

Here is the job description I want to apply for:
<job_description>
{job_description_ts}
</job_description>

Tailor my resume to this job description and return the result as a valid JSON object that strictly follows the schema provided.
"""


In [11]:
llm_message = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [12]:
tailored_data = groq_llm_model(
    api_key= groq_key,
    model=gpt_oss,
    messages=llm_message
)

In [13]:
tailored_data = json.loads(tailored_data)

In [14]:
from builders.docx_builder import Resume_Builder

if tailored_data["sucess"]:
    doc = Resume_Builder(tailored_data["generated_data"])
else:
    print("failed")

  0%|          | 0/1 [00:00<?, ?it/s]

Saved: resume_ark\resume.pdf


In [15]:
display(Markdown(tailored_data["llm_notes"]))

Consider adding a bullet point that mentions any direct experience with Claude Code, Cursor, or Codex (e.g., using LLMs to generate production code). Include a short description of any fintech‑related coursework or a personal project that manipulates financial data or fixed‑income instruments. Highlight exposure to trading concepts or market data pipelines to strengthen alignment with Trumid's domain.

In [16]:
rprint(tailored_data["llm_notes"])

Consider adding a bullet point that mentions any direct experience with Claude Code, Cursor, or Codex (e.g., using 
LLMs to generate production code). Include a short description of any fintech‑related coursework or a personal 
project that manipulates financial data or fixed‑income instruments. Highlight exposure to trading concepts or 
market data pipelines to strengthen alignment with Trumid's domain.